In [1]:
import torch
import torch.nn as nn
# !pip install kagglehub
# import kagglehub
# path = kagglehub.dataset_download("awsaf49/coco-2017-dataset")

# print("Path to dataset files:", path)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"using device: {device}")

using device: mps


In [2]:
# dataset stuff.
import json
with open("coco-2017-dataset/coco2017/annotations/captions_train2017.json") as f:
    data = json.load(f)

In [29]:
# from PIL import Image 
# k_size = 3
# img_size = 100 # 640x640 is normal but not fast and moreso used for detection
# NUM_CLASSES = 80
# # this is a classifier net...
# # not detector.

# class cNet (nn.Module):
#     def __init__(self, num_classes, i_size):
#         s = int((i_size // 4)**2 * 64) # divisions from pooling then * last batch size
#         super().__init__()
#         self.net = nn.Sequential(
#             # input is batch, 3, 100, 100
#             # IF THE CONV PARAMETERS ARE CHANGED YOU MUST UPDATE 's' ABOVE
#             nn.Conv2d(3,16,kernel_size=k_size,padding=1), # thinking small values for faster speed.
#             nn.BatchNorm2d(16),
#             nn.Conv2d(16,32,kernel_size=k_size,padding=1),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(2), # can also do kernel_size = 2 to reduce spatial space.
#             # batch, 32, 50, 50

#             #next large layer
#             nn.Conv2d(32,64,kernel_size=k_size,padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#             # batch batch, 64, 25, 25

#             # output layers
#             #there needs to be a auto fit for the Linear Layer make a variable for it.
#             nn.Flatten(), # batch 64*25*25 = 40000
#             nn.Linear(s, 256), # this size seems a little ridiculous.
#             nn.ReLU(),
#             nn.Linear(256,num_classes)
#         )
#     def forward(self, x):
#         # dataset already resizes via transforms, x is a tensor here not a file path
#         return self.net(x)

In [32]:
# THIS IS A NEW ONE BECAUSE THE OLD MODEL OVERFITS EXTREMELY FAST. 

from PIL import Image 
img_size = 100 # 640x640 is normal but not fast and moreso used for detection
NUM_CLASSES = 80
# this is a classifier net...
# not detector.

#HYPER PARAMETERS
k_size = 3
conv_drop = 0.1
lin_drop = 0.3


class cNet (nn.Module):
    def __init__(self, num_classes, i_size):
        super().__init__()
        self.net = nn.Sequential(
            # input is batch, 3, 100, 100
            # IF THE CONV PARAMETERS ARE CHANGED YOU MUST UPDATE 's' ABOVE
            nn.Conv2d(3,16,kernel_size=k_size,padding=1), # thinking small values for faster speed.
            nn.BatchNorm2d(16),
            nn.Conv2d(16,32,kernel_size=k_size,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout2d(conv_drop), # zeroes out featrure maps in this scenario
            nn.MaxPool2d(2), # can also do kernel_size = 2 to reduce spatial space.
            # batch, 32, 50, 50

            #next large layer
            nn.Conv2d(32,64,kernel_size=k_size,padding=1),
            nn.ReLU(),
            nn.Dropout2d(conv_drop), # again not dropping neurons
            nn.MaxPool2d(2),
            # batch batch, 64, 25, 25

            # the new output layers
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64,256),
            nn.ReLU(),
            nn.Dropout(lin_drop),# dropping neurons
            nn.Linear(256,num_classes)
        )
    def forward(self, x):
        # dataset already resizes via transforms, x is a tensor here not a file path
        return self.net(x)

In [18]:
import os                                                                                                                                                                                 
from torch.utils.data import Dataset                                                                    
import torchvision.transforms as T                                                                      
import json                                                                                             
                                                                                                          
class COCOClassifyDataset(Dataset):                                                                     
    def __init__(self, img_dir, ann_file, img_size=100):                                                
        with open(ann_file) as f:                                                                       
            data = json.load(f)                                                                         
                                                                                                          
        # map category_id -> contiguous index (0 to 79)                                                 
        self.categories = {c['id']: i for i, c in enumerate(data['categories'])}                        
        self.class_names = [c['name'] for c in data['categories']]                                      
                                                                                                          
        # for each image, pick the annotation with the largest area
        best = {}  # image_id -> (area, category_id)                                                    
        for ann in data['annotations']:                                                                 
            iid = ann['image_id']
            if iid not in best or ann['area'] > best[iid][0]:                                           
                best[iid] = (ann['area'], ann['category_id'])
                                                                                                          
        # build flat list of (filename, label)                                                          
        id_to_file = {img['id']: img['file_name'] for img in data['images']}
        self.samples = [                                                                                
            (os.path.join(img_dir, id_to_file[iid]), self.categories[cat_id])
            for iid, (_, cat_id) in best.items()                                                        
            if iid in id_to_file                                                                        
        ]
                                                                                                          
        self.transform = T.Compose([                                                                    
            T.Resize((img_size, img_size)),
            T.ToTensor(),                                                                               
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
        ])                                                                                              
   
    def __len__(self):                                                                                  
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')  # convert handles grayscale images                       
        return self.transform(img), label

In [19]:
from torch.utils.data import DataLoader                                                                 
                  
train_ds = COCOClassifyDataset(
    img_dir='coco-2017-dataset/coco2017/train2017',
    ann_file='coco-2017-dataset/coco2017/annotations/instances_train2017.json'                          
)                                                                                                       
val_ds = COCOClassifyDataset(                                                                           
    img_dir='coco-2017-dataset/coco2017/val2017',                                                       
    ann_file='coco-2017-dataset/coco2017/annotations/instances_val2017.json'
)
# note: COCO test2017 has no annotation file (competition set, labels are withheld)
# use val_loader for final eval instead

# num_workers=0 required in notebooks — worker processes can't import classes defined in cells
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)                         
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)

In [26]:
criterion = nn.CrossEntropyLoss() # loss computer. does softmax and negative log likelihood. less confidence about correct = more loss.

def train(net: nn.Module, epochs=100):
    lr = 3e-4
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) # i like this one

    total_params = sum(p.numel() for p in net.parameters())
    print(f"total trainable parameters: {total_params}\nGood Luck!")

    print("training loop beginning")
    for epoch in range(epochs):
        net.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device) # move batch to mps/cpu
            opt.zero_grad()
            # BIG NOTE so cool thing about pytorch
            # almost every single class from nn records its computation graph so you can do backprop almost anywhere.
            preds = net(images) # get logits  
            loss = criterion(preds, labels) # see how bad
            loss.backward() # take loss and go back
            opt.step() # update weights basically w -= -lr * grad

        scheduler.step() # step dat cosine

        net.eval()
        with torch.no_grad():
            for images, labels in val_loader: # checks based on the val set now
                images, labels = images.to(device), labels.to(device)
                preds = net(images)
                val_loss = criterion(preds, labels)
        if (epoch % 10 == 0):
            print(f"EPOCH: {epoch} train: {loss.item():.4f} | val: {val_loss.item():.4f}")

@torch.no_grad()
def eval(net: nn.Module):
    # get test set this time and test model on that
    net.eval()
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        preds = net(images)
        test_loss = criterion(preds, labels)
        print(f"test loss: {test_loss.item():.4f}")

In [33]:
net = cNet(NUM_CLASSES, img_size).to(device)

In [ ]:
# training here
train(net)

total trainable parameters: 60880
Good Luck!
training loop beginning


In [ ]:
eval(net)

In [ ]:
# save model
torch